In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import numpy as np
import time
from tokenizers import Tokenizer
import torch
import torch.nn as nn
from torch.nn import functional as F
import gc

rng = np.random.default_rng(seed=42)
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

In [2]:
tokenizer_file = '/home/user/Desktop/ml/gpt_dt/tokenizer/tokenizer_02.json'
tokenizer = Tokenizer.from_file(tokenizer_file)

current_step = 0

In [3]:
# evrything here can be changed each training session to optimize learning
batch_size = 32     # effective batch size is minibatch_size * accumulation_steps
block_size = 128    # max possible block size to train
learning_rate = 5e-4
save_iters = 50
eval_iters = 25

# everything below here NEEDS to be identical to load an extistng model
vocab_size = 32768
pad_id = 1
assistant_id = 5
n_embed = 32
n_head = 2
n_layer = 4
dropout = 0.2

In [4]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokens = np.array(tokenizer.encode(text, add_special_tokens=False).ids)
split_idx = int(len(tokens) * 0.9)
train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

def create_batches(data_tokens, batch_size, block_size, rng):
    num_blocks = len(data_tokens) - block_size - 1
    usable_blocks = (num_blocks // batch_size) * batch_size
    indices = rng.permutation(np.arange(num_blocks))[:usable_blocks]
    return indices.reshape(-1, batch_size)

train_batches = create_batches(train_tokens, batch_size, block_size, rng)
val_batches = create_batches(val_tokens, batch_size, block_size, rng)

num_batches = train_batches.shape[0]
print(f"Train batches: {num_batches}")
print(f"Val batches: {val_batches.shape[0]}")

Train batches: 1695
Val batches: 184


In [5]:
def get_batch(split, batch_size, step=None):
    global train_batches, val_batches, train_tokens, val_tokens
    if step is not None and step % num_batches == 0:
        train_batches = create_batches(train_tokens, batch_size, block_size, rng)
        val_batches = create_batches(val_tokens, batch_size, block_size, rng)
        
    data = train_tokens if split == 'train' else val_tokens
    batch_map = train_batches if split == 'train' else val_batches
    
    if step is None:
        ix = batch_map[rng.integers(batch_map.shape[0])]
    else:
        ix = batch_map[step % batch_map.shape[0]]

    x = torch.stack([torch.from_numpy(data[i:i+block_size]) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+block_size+1]) for i in ix])
    return x.to(device), y.to(device)

In [6]:
@torch.no_grad()
def estimate_loss():
    m.eval()
    out = {}
    for split in ['train', 'val']:
        out[split] = 0
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size)
            logits, loss = m.forward(X, Y)
            out[split] += loss
        out[split] /= eval_iters
    m.train()
    return out

In [8]:
def sample_logits(logits, temp=1.0, top_k=0, top_p=0.0, generator=None):
    logits /= temp

    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = float('-inf')

    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Remove tokens with cumulative probability above the threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift indices to keep the first token that exceeds top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[..., indices_to_remove] = float('-inf')

    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1, generator=generator)
    return next_token

In [9]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)
    
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embed, n_head, dropout):
        super().__init__()
        self.n_head = n_head
        self.n_embed = n_embed
        self.dropout_p = dropout
        self.head_size = n_embed // n_head
        self.qkv_projection = nn.Linear(n_embed, n_embed*3, bias=False)
        self.proj = nn.Linear(self.head_size * n_head, n_embed)
        self.dropout = nn.Dropout(dropout)

        # torch.arange(0, D, 2) gets [0, 2, 4, ..., (D - 2)]
        inv_freq = 1.0 / (float(10000.0) ** (torch.arange(0, self.head_size, 2).float() / self.head_size))  # 1 / (base^(2i/D))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_projection(x)
        q, k, v = qkv.split(self.n_embed, dim=2)  # (B, T, C)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, n_head, T, head_size)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        q = q.to(torch.float32)
        k = k.to(torch.float32)

        with torch.amp.autocast(device_type='cuda', enabled=False):
            inv_freq = self.inv_freq.to(torch.float32)
            t = torch.arange(T, device=x.device, dtype=torch.float32)
            
            freqs = torch.einsum("i,j->ij", t, inv_freq) # [T, head_size/2]
            emb = torch.cat((freqs, freqs), dim=-1) # (T, head_size)
            
            cos = emb.cos()[None, None, :, :].to(torch.float32) # [B, nh, T, hs]
            sin = emb.sin()[None, None, :, :].to(torch.float32)
            #print(f"({k[13][12][100][4]}, {k[13][12][100][(head_size//2)+4]})")
            #print(inv_freq[4])
            q = (q * cos) + (rotate_half(q) * sin)
            k = (k * cos) + (rotate_half(k) * sin)
            #print(f"({k[13][12][100][4]}, {k[13][12][100][(head_size//2)+4]})")

        q = q.to(v.dtype) 
        k = k.to(v.dtype)
        
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout_p if self.training else 0.0, is_causal=True)
        
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_embed)
        out = self.dropout(self.proj(out))
        return out
        
class FeedForward(nn.Module):
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed), 
            nn.GELU(approximate='tanh'), 
            nn.Linear(4 * n_embed, n_embed), 
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embed, n_head, dropout):
        super().__init__()
        self.sa = MultiHeadAttention(n_embed, n_head, dropout)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = nn.RMSNorm(n_embed)
        self.ln2 = nn.RMSNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, dropout):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.blocks = nn.ModuleList([Block(n_embed, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            
    def forward(self, index, targets=None):
        B, T = index.shape
        
        x = self.token_embedding_table(index) # (B,T,C)
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens, temp=1.0, top_k=0, top_p=0.0, eot_id=None, seed=None):
        gen = None
        if seed is not None:
            gen = torch.Generator(device=index.device)
            gen.manual_seed(seed)
            
        with torch.no_grad():
            self.eval()
            for _ in range(max_new_tokens):
                # get the logits for each char in the index
                logits, loss = self(index[..., -block_size:])
                # only look at the logits for each last letter
                logits = logits[:, -1, :]
                # choose one sample from our logits
                index_next = sample_logits(logits, temp, top_k, top_p, gen)
                #concatenate the letter choice for each batch onto the end of the existing char list
                index = torch.cat((index, index_next), dim=1) #(B, T+1)
                # stop generating if end of text token is generated
                if eot_id is not None and index_next.item() == eot_id:
                    break
            self.train()
        return index

In [10]:
torch.manual_seed(70)
m = GPTLanguageModel(vocab_size, n_embed, n_head, n_layer, dropout).to(device)

def get_optimizer_groups(model, weight_decay):
    """ Separates parameters into decay and no-decay groups """
    decay = set()
    no_decay = set()
    whitelist_weight_modules = (torch.nn.Linear, torch.nn.Embedding)
    blacklist_weight_modules = (torch.nn.LayerNorm, torch.nn.RMSNorm)

    for mn, m in model.named_modules():
        for pn, p in m.named_parameters():
            fpn = f"{mn}.{pn}" if mn else pn # full param name
            
            if pn.endswith('bias'):
                # all biases will not be decayed
                no_decay.add(fpn)
            elif pn.endswith('weight') and isinstance(m, whitelist_weight_modules):
                # weights of linear/embedding modules will be decayed
                decay.add(fpn)
            elif pn.endswith('weight') and isinstance(m, blacklist_weight_modules):
                # weights of layernorm/rmsnorm will not be decayed
                no_decay.add(fpn)
            elif pn.endswith('g'): # For custom NumPy-style names
                no_decay.add(fpn)

    # Validate that we didn't miss anything
    param_dict = {pn: p for pn, p in model.named_parameters()}
    inter_params = decay & no_decay
    union_params = decay | no_decay
    assert len(inter_params) == 0, "Parameters appearing in both groups!"
    assert len(param_dict.keys() - union_params) == 0, "Missing parameters in grouping!"

    # Create the actual group objects
    optim_groups = [
        {"params": [param_dict[pn] for pn in sorted(list(decay))], "weight_decay": weight_decay},
        {"params": [param_dict[pn] for pn in sorted(list(no_decay))], "weight_decay": 0.0},
    ]
    return optim_groups
    
# Create the optimizer with grouped parameters
optim_groups = get_optimizer_groups(m, weight_decay=0.01)
optim = torch.optim.AdamW(optim_groups, lr=learning_rate) #

#optim = torch.optim.AdamW(m.parameters(), lr=learning_rate)
params = sum(p.numel() for p in m.parameters())
print(f"Total parameters: {params:,}")

Total parameters: 2,180,128


In [11]:
max_iters = 10000

In [ ]:
losses = estimate_loss()    # estimate a base loss before training session
print(f"Step: {current_step:04d}, Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")

times_tracker = {'train': {'time': 0.0, 'count': 0}, 'estimate': {'time': 0.0, 'count': 0}}
tic = time.perf_counter()
for step in range(current_step+1, max_iters + 1):
    optim.zero_grad(set_to_none=True)
    xb, yb = get_batch('train', batch_size, step)
    logits, loss = m.forward(xb, yb)
    loss.backward()
    optim.step()
    current_step = step
    if step % save_iters == 0:
        times_tracker['train']['time'] += (time.perf_counter() - tic)
        times_tracker['train']['count'] += save_iters
        
        print()
        tic = time.perf_counter()
        losses = estimate_loss()
        times_tracker['estimate']['time'] += (time.perf_counter() - tic)
        times_tracker['estimate']['count'] += 1
        print(f"Step: {step:04d}, Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")
        tic = time.perf_counter()
wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/user/.netrc.
wandb: Currently logged in as: louieluri (laluriinc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step: 0000, Train Loss: 10.4101, Val Loss: 10.4075

Step: 0050, Train Loss: 9.0984, Val Loss: 9.1761

Step: 0100, Train Loss: 7.9997, Val Loss: 8.1407

Step: 0150, Train Loss: 7.2393, Val Loss: 7.4278

Step: 0200, Train Loss: 6.7484, Val Loss: 6.9989

Step: 0250, Train Loss: 6.4322, Val Loss: 6.7097

Step: 0300, Train Loss: 6.1955, Val Loss: 6.5384

Step: 0350, Train Loss: 5.9759, Val Loss: 6.3491

Step: 0400, Train Loss: 5.7525, Val Loss: 6.1586

Step: 0450, Train Loss: 5.5261, Val Loss: 6.0041

Step: 0500, Train Loss: 5.3063, Val Loss: 5.8251

Step: 0550, Train Loss: 5.0764, Val Loss: 5.6847

Step: 0600, Train Loss: 4.8753, Val Loss: 5.5362

Step: 0650, Train Loss: 4.6626, Val Loss: 5.4446

Step: 0700, Train Loss: 4.4972, Val Loss: 5.3563

Step: 0750, Train Loss: 4.3514, Val Loss: 5.2662

Step: 0800, Train Loss: 4.2252, Val Loss: 5.2310

Step: 0850, Train Loss: 4.0876, Val Loss: 5.1742

Step: 0900, Train Loss: 3.9659, Val Loss: 5.1291

Step: 0950, Train Loss: 3.8296, Val Loss: 5.1215

avg_time_estimate,▁▁▁▂▂▃▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██
avg_time_step,█▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss_train,█▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss_val,▅▄▃▁▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇████████
step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
avg_time_estimate,0.2444
avg_time_step,0.01216
loss_train,0.82969
loss_val,10.29939
step,10000


In [12]:
print(f"Total time train: {times_tracker['train']['time'] /60:.3f}min")
print(f"Total time estim: {times_tracker['estimate']['time'] /60:.3f}min")
print(f"Average time per Optimizer step: {times_tracker['train']['time'] / times_tracker['train']['count']:.3f}sec")
print(f"Average time {save_iters} Optimizr steps: {times_tracker['train']['time'] / (times_tracker['train']['count'] / save_iters):.3f}sec")
print(f"Average time per Estimate  Loss: {times_tracker['estimate']['time'] / times_tracker['estimate']['count']:.3f}sec")

Total time train: 4.042min
Total time estim: 1.570min
Average time per Optimizer step: 0.016sec
Average time 50 Optimizr steps: 0.808sec
Average time per Estimate  Loss: 0.314sec


In [15]:
print(tokenizer.decode(m.generate(torch.tensor([tokenizer.encode("I believe, my dears, that I am the proudest story-teller", add_special_tokens=False).ids]).to(device), 512, temp=0.8, top_k=50, top_p=0.9).cpu().tolist()[0]))

I believe, my dears, that I am the proudest story-teller that ever
lived. Many a time tears of pride and joy have stood in my eyes while I
read the tender, loving, appealing letters that come to me in almost
every mail from my little readers. To have pleased you, to have
interested you, to have won your friendship, and perhaps your love,
through my stories, is to my mind as great an achievement as to become
President of the United States. Indeed, I would much rather be your
story-teller, under these conditions, than to be the President. So you
have helped me to fulfill my life's ambition, and I am more grateful to
you, my dears, than I can express in words.

I try to answer every letter of my young correspondents; yet sometimes
there are so many letters that a little time must pass before you get
your answer. But be patient, friends, for the answer will surely come,
and by writing to me you more than repay me for the pleasant task of
preparing these books. Besides, I am proud to acknow

In [13]:
def save_pt_to_numpy(model, filename="model_weights.npz"):
    weights_dict = {}
    
    # Embeddings
    weights_dict['tok_emb'] = model.token_embedding_table.weight.data.cpu().numpy()
    
    # Blocks
    for i, block in enumerate(model.blocks):
        # Attention - Transpose Linear weights for your (x @ W) format
        weights_dict[f'b{i}_qkv_w'] = block.sa.qkv_projection.weight.data.cpu().numpy().T
        weights_dict[f'b{i}_proj_w'] = block.sa.proj.weight.data.cpu().numpy().T
        weights_dict[f'b{i}_proj_b'] = block.sa.proj.bias.data.cpu().numpy()
        
        # RMSNorm
        weights_dict[f'b{i}_ln1_g'] = block.ln1.weight.data.cpu().numpy()
        weights_dict[f'b{i}_ln2_g'] = block.ln2.weight.data.cpu().numpy()
        
        # MLP (FeedForward)
        weights_dict[f'b{i}_ff_w1'] = block.ffwd.net[0].weight.data.cpu().numpy().T
        weights_dict[f'b{i}_ff_b1'] = block.ffwd.net[0].bias.data.cpu().numpy()
        
        weights_dict[f'b{i}_ff_w2'] = block.ffwd.net[2].weight.data.cpu().numpy().T
        weights_dict[f'b{i}_ff_b2'] = block.ffwd.net[2].bias.data.cpu().numpy()

    # Head and Final Norm
    weights_dict['ln_f_g'] = model.ln_f.weight.data.cpu().numpy()
    weights_dict['head_w'] = model.lm_head.weight.data.cpu().numpy().T
    weights_dict['head_b'] = model.lm_head.bias.data.cpu().numpy()

    np.savez(filename, **weights_dict)
    print(f"Weights saved to {filename}")
save_pt_to_numpy(m)

Weights saved to model_weights.npz


In [10]:
def check_pt_gradients(model, step):
    print(f"\n=== PyTorch Gradient Check {step} ===")
    
    # 1. LM Head
    print(f"LM Head dW Max: {model.lm_head.weight.grad.abs().max().item():.8e}")
    print(f"LM Head db Max: {model.lm_head.bias.grad.abs().max().item():.8e}")
    
    # 2. Final Norm
    print(f"Final LN dg Max: {model.ln_f.weight.grad.abs().max().item():.8e}")
    
    # 3. Block FeedForward (Indices: 2 is Linear2, 0 is Linear1)
    ff = model.blocks[-1].ffwd
    print(f"FF Linear2 dW Max: {ff.net[2].weight.grad.abs().max().item():.8e}")
    print(f"FF Linear2 db Max: {ff.net[2].bias.grad.abs().max().item():.8e}")
    print(f"FF Linear1 dW Max: {ff.net[0].weight.grad.abs().max().item():.8e}")
    print(f"FF Linear1 db Max: {ff.net[0].bias.grad.abs().max().item():.8e}")

    print(f"LN2 (FF) dg Max: {model.blocks[-1].ln2.weight.grad.abs().max().item():.8e}")
    
    # 4. Attention
    sa = model.blocks[-1].sa
    print(f"SA Proj dW Max: {sa.proj.weight.grad.abs().max().item():.8e}")
    print(f"SA Proj db Max: {sa.proj.bias.grad.abs().max().item():.8e}")
    print(f"SA QKV  dW Max: {sa.qkv_projection.weight.grad.abs().max().item():.8e}")

    print(f"LN1 (SA) dg Max: {model.blocks[-1].ln1.weight.grad.abs().max().item():.8e}")

In [11]:
losses = estimate_loss()    # estimate a base loss before training session
print(f"Step: {current_step:04d}, Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")

for step in range(current_step+1, 2 + 1):
    optim.zero_grad(set_to_none=True)
    xb, yb= get_batch('train', batch_size, step)
    logits, loss = m.forward(xb, yb)
    print(f"\nLoss: {loss}")
    loss.backward()
    check_pt_gradients(m, step)
    optim.step()

Step: 0000, Train Loss: 10.3169, Val Loss: 10.3167

Loss: 10.315443992614746

=== PyTorch Gradient Check 1 ===
LM Head dW Max: 3.50640751e-02
LM Head db Max: 4.56220657e-02
Final LN dg Max: 1.97040802e-03
FF Linear2 dW Max: 5.60607715e-03
FF Linear2 db Max: 6.80886358e-02
FF Linear1 dW Max: 5.25190728e-03
FF Linear1 db Max: 6.93726260e-03
LN2 (FF) dg Max: 6.01763546e-04
SA Proj dW Max: 6.51901355e-03
SA Proj db Max: 6.07213229e-02
SA QKV  dW Max: 6.85239816e-03
LN1 (SA) dg Max: 5.71045442e-04

Loss: 10.236490249633789

=== PyTorch Gradient Check 2 ===
LM Head dW Max: 7.54179060e-02
LM Head db Max: 4.51248512e-02
Final LN dg Max: 7.64639629e-03
FF Linear2 dW Max: 7.75643997e-03
FF Linear2 db Max: 3.85954008e-02
FF Linear1 dW Max: 6.49493653e-03
FF Linear1 db Max: 3.96090187e-03
LN2 (FF) dg Max: 1.01986248e-03
SA Proj dW Max: 1.05213784e-02
SA Proj db Max: 3.69733870e-02
SA QKV  dW Max: 1.01477280e-02
LN1 (SA) dg Max: 1.24831754e-03
